In [1]:
import pandas as pd
import os
os.getcwd()

'/Users/tyrondelatorre/Desktop/work/Portfolio/Real Estate Investment/Data Cleaning and EDA'

## 1. Data import and initial inspection of the dataset
This first step consists of importing the dataset and performing an initial inspection to understand its structure and general characteristics.

The inspection is carried out using functions such as ***.head()*** and ***.info()***, which allow to quickly assess the number of columns, the type of variables, and the overall completeness of the data.

The dataset is composed of 20 columns (as shown in the related output) and 240 records.

In [3]:
dt = pd.read_csv("../Datasets/investement_df.csv")

In [4]:
dt.head(5)

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N),Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Superficie (L/N).1,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente,Destinazione,Universities,Proximity,Type,Tipologia prevalente
0,Abitazioni civili,NORMALE,2800,4000,L,103,148,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
1,Abitazioni di tipo economico,NORMALE,2550,3500,L,9,128,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
2,Box,NORMALE,1300,1850,L,63,93,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
3,Posti auto coperti,NORMALE,1000,1400,L,53,75,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN
4,Posti auto scoperti,NORMALE,550,800,L,33,48,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public,NaN


In [5]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Tipologia                              240 non-null    object
 1   Stato conservativo                     240 non-null    object
 2   Valori Compravendita (€/mq) - Min      240 non-null    int64 
 3   Valori Compravendita (€/mq) - Max      240 non-null    int64 
 4   Superficie (L/N)                       240 non-null    object
 5   Valori Locazione (€/mq x mese) - Min   240 non-null    int64 
 6   Valori Locazione (€/mq x mese) - Max   240 non-null    int64 
 7   Superficie (L/N) - Superficie (L/N).1  240 non-null    object
 8   Area                                   240 non-null    object
 9   Province                               240 non-null    object
 10  Municipality                           240 non-null    object
 11  Fascia/Zona        

## 2. Consistency checks, validation, and column name correction
The initial inspection highlighted inconsistencies in column naming. In particular, two columns referring to the same feature were present due to an error in the naming process.

In addition, some columns (in particular those related to surface) were not named clearly enough to distinguish between different contexts, even though they contained different data.

Before applying the corrections, a few validation checks were also performed to better understand the dataset and identify possible data quality issues. The .describe() function was used on the numerical columns to inspect the main statistics and manually check whether the minimum values were lower than the corresponding maximum values. A separate check was also performed to verify the presence of zero or negative values in the price columns, since these would not make sense in this context.

Duplicate records were also checked to confirm whether identical rows were present in the dataset.

A consistency check on selected categorical columns was then performed to identify potential mismatches in string values (e.g. different capitalizations or spellings for the same category).

This step therefore consists of:
- checking for duplicated records
- checking descriptive statistics for the numerical columns
- checking for zero or negative values in the price columns
- merging columns referring to the same feature by filling missing values from one column into the other
- removing redundant columns
- clarifying ambiguous column names to better reflect the underlying data
- correcting inconsistencies in categorical values

A second check using ***.info()***, ***.isna().any()***, and ***.unique()*** (applied to the columns containing inconsistencies) is then performed to confirm that the transformations were applied correctly.

In [11]:
dt.duplicated().sum()

0

In [23]:
price_cols = [
    "Valori Compravendita (€/mq) - Min",
    "Valori Compravendita (€/mq) - Max",
    "Valori Locazione (€/mq x mese) - Min",
    "Valori Locazione (€/mq x mese) - Max"
]

dt[price_cols].describe()

,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max
count,240.000000,240.000000,240.000000,240.000000
mean,2239.166667,3180.000000,68.258333,92.433333
std,1375.414480,1913.811286,45.732714,69.901872
min,400.000000,600.000000,3.000000,4.000000
25%,1150.000000,1687.500000,21.250000,17.750000
50%,2100.000000,2975.000000,68.000000,88.000000
75%,2900.000000,4025.000000,103.000000,143.000000
max,8300.000000,11000.000000,195.000000,288.000000


In [53]:
print("Is there any negative value?")
print()
(dt[price_cols] <= 0).any()

Is there any negative value?



Valori Compravendita (€/mq) - Min       False
Valori Compravendita (€/mq) - Max       False
Valori Locazione (€/mq x mese) - Min    False
Valori Locazione (€/mq x mese) - Max    False
dtype: bool

In [13]:
dt["Tipologia Prevalente"] = dt["Tipologia Prevalente"].fillna(dt["Tipologia prevalente"])
dt.drop(columns = "Tipologia prevalente", inplace = True)

dt.rename( columns  = {
    "Superficie (L/N)": "Superficie (L/N) - Compravendita",
    "Superficie (L/N) - Superficie (L/N).1": "Superficie (L/N) - Locazione"
}, inplace = True)
dt.head(1)

,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N) - Compravendita,Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Superficie (L/N) - Locazione,Area,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente,Destinazione,Universities,Proximity,Type
0,Abitazioni civili,NORMALE,2800,4000,L,103,148,L,San Paolo,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale,RM3,Core,public


In [57]:
columns = ["Tipologia", "Stato conservativo", "Superficie (L/N) - Compravendita", "Superficie (L/N) - Locazione",
      "Area", "Fascia/Zona", "Province", "Municipality", "Tipologia Prevalente", "Destinazione"]

for col in columns:
    print(f"The unique values for {col} are: ")
    print(dt[col].unique().tolist())
    print()
    print(f"The number of unique values for this column are:")
    print(dt[col].nunique())
    print("-----------------------------------------------")
    print()

The unique values for Tipologia are: 
['Abitazioni civili', 'Abitazioni di tipo economico', 'Box', 'Posti auto coperti', 'Posti auto scoperti', 'Abitazioni signorili', 'Ville e Villini']

The number of unique values for this column are:
7
-----------------------------------------------

The unique values for Stato conservativo are: 
['NORMALE']

The number of unique values for this column are:
1
-----------------------------------------------

The unique values for Superficie (L/N) - Compravendita are: 
['L']

The number of unique values for this column are:
1
-----------------------------------------------

The unique values for Superficie (L/N) - Locazione are: 
['L']

The number of unique values for this column are:
1
-----------------------------------------------

The unique values for Area are: 
['San Paolo', 'Testaccio', 'Ostiense', 'Garbatella', 'Marconi', 'San Lorenzo', 'Monteverde Nuovo', 'Monteverde Vecchio', 'Parioli', 'Trastevere', 'Pigneto', 'Nomentano', 'Re di Roma', 'Pi

In [15]:
dt["Tipologia Prevalente"] = dt["Tipologia Prevalente"].replace({"Abitazioni Civili": "Abitazioni civili",
                                                                 "Abitazioni di Tipo Economico": "Abitazioni di tipo economico"})

In [17]:
dt["Tipologia Prevalente"].unique()

array(['Abitazioni di tipo economico', 'Abitazioni civili'], dtype=object)

In [19]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Tipologia                             240 non-null    object
 1   Stato conservativo                    240 non-null    object
 2   Valori Compravendita (€/mq) - Min     240 non-null    int64 
 3   Valori Compravendita (€/mq) - Max     240 non-null    int64 
 4   Superficie (L/N) - Compravendita      240 non-null    object
 5   Valori Locazione (€/mq x mese) - Min  240 non-null    int64 
 6   Valori Locazione (€/mq x mese) - Max  240 non-null    int64 
 7   Superficie (L/N) - Locazione          240 non-null    object
 8   Area                                  240 non-null    object
 9   Province                              240 non-null    object
 10  Municipality                          240 non-null    object
 11  Fascia/Zona                     

In [21]:
dt.isna().any()

Tipologia                               False
Stato conservativo                      False
Valori Compravendita (€/mq) - Min       False
Valori Compravendita (€/mq) - Max       False
Superficie (L/N) - Compravendita        False
Valori Locazione (€/mq x mese) - Min    False
Valori Locazione (€/mq x mese) - Max    False
Superficie (L/N) - Locazione            False
Area                                    False
Province                                False
Municipality                            False
Fascia/Zona                             False
Codice Zona                             False
Microzona Catastale                     False
Tipologia Prevalente                    False
Destinazione                            False
Universities                            False
Proximity                               False
Type                                    False
dtype: bool

## 3. Column reordering
Some columns contain more relevant information for the analysis, in particular those related to the area and the universities, together with their associated attributes.

The goal of this step is to improve the readability of the dataset by grouping related variables and placing the most informative columns in more prominent positions.

This step is not strictly necessary for the analysis, but is included to improve readability and to demonstrate control over the structure of the dataset.

In a following step, some columns will be deleted either because they contain constant values or because they do not provide relevant information for the analysis.

In [24]:
col_list = dt.columns.tolist()
col_list

['Tipologia',
 'Stato conservativo',
 'Valori Compravendita (€/mq) - Min',
 'Valori Compravendita (€/mq) - Max',
 'Superficie (L/N) - Compravendita',
 'Valori Locazione (€/mq x mese) - Min',
 'Valori Locazione (€/mq x mese) - Max',
 'Superficie (L/N) - Locazione',
 'Area',
 'Province',
 'Municipality',
 'Fascia/Zona',
 'Codice Zona',
 'Microzona Catastale',
 'Tipologia Prevalente',
 'Destinazione',
 'Universities',
 'Proximity',
 'Type']

In [26]:
col_list.remove("Area")
col_list.insert(0, "Area")
col_list.remove("Universities")
col_list.insert(8, "Universities")
col_list.remove("Proximity")
col_list.insert(9, "Proximity")
col_list.remove("Type")
col_list.insert(10, "Type")
dt = dt[col_list]
dt

,Area,Tipologia,Stato conservativo,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Superficie (L/N) - Compravendita,Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Universities,Proximity,Type,Superficie (L/N) - Locazione,Province,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente,Destinazione
0,San Paolo,Abitazioni civili,NORMALE,2800,4000,L,103,148,RM3,Core,public,L,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale
1,San Paolo,Abitazioni di tipo economico,NORMALE,2550,3500,L,9,128,RM3,Core,public,L,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale
2,San Paolo,Box,NORMALE,1300,1850,L,63,93,RM3,Core,public,L,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale
3,San Paolo,Posti auto coperti,NORMALE,1000,1400,L,53,75,RM3,Core,public,L,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale
4,San Paolo,Posti auto scoperti,NORMALE,550,800,L,33,48,RM3,Core,public,L,Rome,Rome,Periferica/San Paolo (Tullio Levi Civita),D5,53,Abitazioni di tipo economico,Residenziale
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,Centocelle,Posti auto scoperti,NORMALE,600,900,L,33,48,SAP,extended,public,L,Rome,Rome,Periferica/CENTOCELLE (PIAZZA DEI MIRTI),D14,111,Abitazioni di tipo economico,Residenziale
236,Tintoretto,Abitazioni civili,NORMALE,2450,3400,L,105,15,"RM3, TVU","nearby, extended","public, public",L,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni civili,Residenziale
237,Tintoretto,Box,NORMALE,1350,1900,L,6,85,"RM3, TVU","nearby, extended","public, public",L,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni civili,Residenziale
238,Tintoretto,Posti auto coperti,NORMALE,1000,1450,L,53,75,"RM3, TVU","nearby, extended","public, public",L,Rome,Rome,Periferica/TINTORETTO (VIA BALLARIN),D38,122,Abitazioni civili,Residenziale


## 4. Filtering irrelevant records
After an initial check of the different property types, some categories are excluded from the dataset as they are not relevant for the scope of the analysis.

In particular, since the analysis focuses on residential properties, entries related to parking spaces (e.g. box, covered and uncovered parking) are removed, as they are not directly comparable in terms of pricing.

A final check of the remaining categories is performed to confirm that only relevant property types are retained.

In [29]:
dt["Tipologia"].unique().tolist()

['Abitazioni civili',
 'Abitazioni di tipo economico',
 'Box',
 'Posti auto coperti',
 'Posti auto scoperti',
 'Abitazioni signorili',
 'Ville e Villini']

In [31]:
dt = dt[(dt["Tipologia"] != "Posti auto scoperti") & (
    dt["Tipologia"] != "Posti auto coperti") & (
    dt["Tipologia"] != "Box")]
dt = dt.reset_index(drop = True)

In [33]:
dt["Tipologia"].unique().tolist()

['Abitazioni civili',
 'Abitazioni di tipo economico',
 'Abitazioni signorili',
 'Ville e Villini']

## 5. Removing non-informative columns
As mentioned before, some columns do not provide useful information as they contain the same value across all records. After verifying that these columns are constant, they are removed to keep the dataset as simple and light as possible.

This is the case of:

- ***Stato Conservativo***: the OMI dataset reports the most common conservation type for the area. In this specific dataset, every area has the same conservation type ("Normale").
- Both columns with ***Superficie*** indicate whether the prices refer to net ("N") or gross ("L") surface. In the present dataset, all records refer to gross surface.
- ***Province***: this column is constant across the dataset, as all properties are located in the municipality of Rome, which is also within the Rome province.
- ***Destinazione***: this column does not add further information, as all remaining records refer to residential properties (after the previous filtering step).

**Note**: feature names and categorical values will be normalized and translated into English in a later step.

In [36]:
dt[["Stato conservativo", "Superficie (L/N) - Compravendita", "Superficie (L/N) - Locazione", "Province", "Destinazione"]].apply(lambda col: col.unique())

,Stato conservativo,Superficie (L/N) - Compravendita,Superficie (L/N) - Locazione,Province,Destinazione
0,NORMALE,L,L,Rome,Residenziale


In [38]:
check = ["Stato conservativo", "Superficie (L/N) - Compravendita", "Superficie (L/N) - Locazione", "Province", "Destinazione"]
for col in check:
    print(col)
    print(dt[col].nunique())
    print(dt[col].unique())
    print()

Stato conservativo
1
['NORMALE']

Superficie (L/N) - Compravendita
1
['L']

Superficie (L/N) - Locazione
1
['L']

Province
1
['Rome']

Destinazione
1
['Residenziale']



In [40]:
dt.drop(columns = check, inplace = True)
dt.head(0)

,Area,Tipologia,Valori Compravendita (€/mq) - Min,Valori Compravendita (€/mq) - Max,Valori Locazione (€/mq x mese) - Min,Valori Locazione (€/mq x mese) - Max,Universities,Proximity,Type,Municipality,Fascia/Zona,Codice Zona,Microzona Catastale,Tipologia Prevalente


## 6. Name normalization
This step was introduced at a later stage to improve the readability and usability of the dataset.

Column names and categorical values are standardized and translated into English, making the dataset more consistent and easier to interpret, especially in the context of sharing results or presenting the analysis.

Although not strictly required for the analysis itself, this step enhances clarity and ensures the dataset is more accessible for future use.

A final check is performed to confirm that both column names and values have been correctly updated.

In [43]:
dt.columns.tolist()

['Area',
 'Tipologia',
 'Valori Compravendita (€/mq) - Min',
 'Valori Compravendita (€/mq) - Max',
 'Valori Locazione (€/mq x mese) - Min',
 'Valori Locazione (€/mq x mese) - Max',
 'Universities',
 'Proximity',
 'Type',
 'Municipality',
 'Fascia/Zona',
 'Codice Zona',
 'Microzona Catastale',
 'Tipologia Prevalente']

In [45]:
dt.rename(columns = {
    "Area": "Area",
    "Tipologia": "Property Type",
    "Valori Compravendita (€/mq) - Min": "Sale Price €/sqm Min",
    "Valori Compravendita (€/mq) - Max": "Sale Price €/sqm Max",
    "Valori Locazione (€/mq x mese) - Min": "Rent Price €/sqm per Month Min",
    "Valori Locazione (€/mq x mese) - Max": "Rent Price €/sqm per Month Max",
    "Universities": "Nearby Universities",
    "Proximity": "University Proximity",
    "Type": "University Type",
    "Fascia/Zona": "OMI Zone",
    "Codice Zona": "OMI Zone Code",
    "Microzona Catastale": "Cadastral Microzone",
    "Tipologia Prevalente": "Dominant Property Type"
}, inplace = True)
dt.head(0)

,Area,Property Type,Sale Price €/sqm Min,Sale Price €/sqm Max,Rent Price €/sqm per Month Min,Rent Price €/sqm per Month Max,Nearby Universities,University Proximity,University Type,Municipality,OMI Zone,OMI Zone Code,Cadastral Microzone,Dominant Property Type


In [48]:
dt["Property Type"].unique().tolist(), dt["Dominant Property Type"].unique().tolist()

(['Abitazioni civili',
  'Abitazioni di tipo economico',
  'Abitazioni signorili',
  'Ville e Villini'],
 ['Abitazioni di tipo economico', 'Abitazioni civili'])

In [50]:
dt["Property Type"].replace({
    "Abitazioni civili": "Civil residential",
    "Abitazioni di tipo economico": "Economic residential",
    "Abitazioni signorili": "Luxury residential",
    "Ville e Villini": "Villas and small villas"
}, inplace = True)

dt["Dominant Property Type"].replace({
    "Abitazioni civili": "Civil residential",
    "Abitazioni di tipo economico": "Economic residential"
}, inplace = True)

/var/folders/gs/d24257s15q1bzbn_vyvcxhb00000gp/T/ipykernel_58267/3675412850.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dt["Property Type"].replace({
/var/folders/gs/d24257s15q1bzbn_vyvcxhb00000gp/T/ipykernel_58267/3675412850.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a cop

In [52]:
dt[["Property Type", "Dominant Property Type"]]

,Property Type,Dominant Property Type
0,Civil residential,Economic residential
1,Economic residential,Economic residential
2,Civil residential,Civil residential
3,Economic residential,Civil residential
4,Civil residential,Civil residential
...,...,...
106,Civil residential,Civil residential
107,Economic residential,Civil residential
108,Civil residential,Economic residential
109,Economic residential,Economic residential


## 7. Exporting the Dataset to CSV file
After completing the cleaning and preparation steps, the final dataset is exported to a CSV file. This allows the dataset to be reused for further analysis or imported into other tools (e.g. Power BI) for visualization and reporting.

In [79]:
dt.to_csv("dt_rome_omi_final.csv", index = False) #OMI_RM_Cleaned